In [0]:
%pip install networkx scikit-learn sentence-transformers

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# Sample data for testing skillgap.py functions
import importlib
import pandas as pd
import networkx as nx

# IMPORTANT: Reload the module to pick up any fixes
import skillgap.skillgap
importlib.reload(skillgap.skillgap)

# FIX: Inject databricks environment into the module's namespace
skillgap.skillgap.spark = spark
skillgap.skillgap.dbutils = dbutils
skillgap.skillgap.IN_DATABRICKS = True
skillgap.skillgap.USE_SQL_CONNECTOR = False  # Force Spark SQL mode in notebook

from skillgap.skillgap import (
    load_kg_output_for_role,
    load_jd_demand_scores,
    load_peer_data,
    build_knowledge_graph, 
    find_skill_gaps, 
    arbitrate_skill_gaps
)

current_role = "Software System Tester"
target_role = "Software Developer"  # Changed to a role with available data

# User's current skills
user_skills = ["Testing", "Bug Reporting", "Automation", "Scripting"]

print(f"Loading knowledge graph data for: {target_role}")

# Load ACTUAL role-specific data from knowledge graph
kg_df = load_kg_output_for_role(target_role)

if len(kg_df) == 0:
    print(f"⚠️ No knowledge graph data found for '{target_role}'.")
    print("This could mean:")
    print("  1. The role doesn't exist in the knowledge graph table")
    print("  2. The table is empty or not accessible")
    print("  3. Try a different role name (e.g., 'Data Analyst', 'Software Engineer')")
else:
    print(f"✅ Found {len(kg_df)} skills for {target_role}")
    print(f"Skills: {kg_df['skill_name'].tolist()[:10]}...")  # Show first 10
    
    # Build knowledge graph
    graph = build_knowledge_graph(target_role, kg_df)
    
    # Find skill gaps
    skill_gaps = find_skill_gaps(user_skills, target_role, graph)
    
    # Load demand scores and peer data
    role_required_skills, jd_demand_db = load_jd_demand_scores(target_role, kg_df)
    peer_data, _ = load_peer_data(target_role)
    
    # Arbitrate skill gaps
    arbitrated_gaps = arbitrate_skill_gaps(skill_gaps, user_skills, graph, jd_demand_db, role_required_skills, peer_data)
    
    print(f"\n{'='*60}")
    print(f"Skill gaps for {target_role}:")
    print(f"{'='*60}")
    if len(arbitrated_gaps) == 0:
        print("✅ No skill gaps found! You already have all required skills.")
    else:
        for gap in arbitrated_gaps:
            print(f"  {gap['skill']}: priority={gap['priority']}, score={gap['unified_score']:.2f}")